### Helper functions 

In [1]:
from collections import deque
import csv
from datetime import datetime, timedelta
import math
import os
import random


def inspect_source_csv_by_date(
    source_path: str, update_fraction: float, datetime_builder: callable
):
    source_row_count = 0
    latest_dt = None

    with open(source_path, "r", newline="", encoding="utf-8-sig") as source_file:
        reader = csv.DictReader(source_file)
        source_header = reader.fieldnames

        for row in reader:
            source_row_count += 1
            row_dt = datetime_builder(row)
            if row_dt is not None:
                latest_dt = row_dt if latest_dt is None else max(latest_dt, row_dt)

    target_hours = math.ceil(8760 * update_fraction)
    cutoff_dt = latest_dt - timedelta(hours=target_hours)

    source_tail = []

    with open(source_path, "r", newline="", encoding="utf-8-sig") as source_file:
        reader = csv.DictReader(source_file)
        for row in reader:
            row_dt = datetime_builder(row)
            if row_dt and row_dt > cutoff_dt:
                source_tail.append(row)

    return source_header, source_row_count, source_tail, latest_dt, cutoff_dt


def write_incremental_csv(update_path: str, update_header: list, rows: list):
    os.makedirs(os.path.dirname(update_path), exist_ok=True)

    with open(update_path, "w", newline="", encoding="utf-8") as update_file:
        update_writer = csv.DictWriter(update_file, fieldnames=update_header)
        update_writer.writeheader()
        update_writer.writerows(rows)


def generate_incremental_update(
    source_path: str,
    update_path: str,
    update_fraction: float,
    datetime_builder: callable,
    datetime_formatter: callable,
    transform_row: callable = None,
    extra_columns: list = None,
    time_step: timedelta = timedelta(hours=1),
    duplicate_fraction: float = 0.0,
):
    (
        source_header,
        source_row_count,
        source_tail,
        latest_dt,
        cutoff_dt,
    ) = inspect_source_csv_by_date(source_path, update_fraction, datetime_builder)

    if not source_tail or latest_dt is None:
        raise ValueError(f"No valid records or timestamps found in {source_path}")

    target_start_dt = latest_dt + time_step
    time_shift = target_start_dt - cutoff_dt - time_step

    generated_rows = []
    for source_row in source_tail:
        row_dict = dict(source_row)
        orig_dt = datetime_builder(row_dict)

        if orig_dt is not None:
            new_dt = orig_dt + time_shift
            row_dict = datetime_formatter(row_dict, new_dt, time_shift)

        if transform_row is not None:
            row_dict = transform_row(row_dict)

        generated_rows.append(row_dict)

    num_duplicates = math.ceil(len(generated_rows) * duplicate_fraction)
    duplicate_rows = random.sample(source_tail, num_duplicates) if num_duplicates > 0 else []

    all_output_rows = generated_rows + [dict(r) for r in duplicate_rows]

    update_header = list(source_header) + list(extra_columns or [])
    write_incremental_csv(update_path, update_header, all_output_rows)

    period_start = datetime_builder(generated_rows[0])
    period_end = datetime_builder(generated_rows[-1])

    return (
        source_row_count,
        len(generated_rows),
        num_duplicates,
        period_start,
        period_end,
    )

### Air quality

In [2]:
from datetime import datetime

aqi_breakpoints = [
    (0.0, 9.0, 0, 50),
    (9.1, 35.4, 51, 100),
    (35.5, 55.4, 101, 150),
    (55.5, 125.4, 151, 200),
    (125.5, 225.4, 201, 300),
    (225.5, 500.4, 301, 500),
]


def pm25_to_aqi(measurement):
    if measurement is None or str(measurement).strip() == "":
        return ""
    try:
        concentration = float(measurement)
        # EPA standard: clamp negative noise/sensor drift to 0.0
        concentration = max(0.0, concentration)
    except (TypeError, ValueError):
        return ""

    for concentration_low, concentration_high, aqi_low, aqi_high in aqi_breakpoints:
        if concentration <= concentration_high:
            aqi = ((aqi_high - aqi_low) / (concentration_high - concentration_low)) * (
                concentration - concentration_low
            ) + aqi_low
            return str(round(aqi))

    return "0"


def air_quality_dt_builder(row):
    try:
        dt_str = f"{row['Date Local']} {row['Time Local']}"
        return datetime.strptime(dt_str, "%Y-%m-%d %H:%M")
    except (KeyError, ValueError):
        return None


def air_quality_dt_formatter(row, new_dt, time_shift):
    row["Date Local"] = new_dt.strftime("%Y-%m-%d")
    row["Time Local"] = new_dt.strftime("%H:%M")

    if "Date GMT" in row and "Time GMT" in row and row["Date GMT"]:
        try:
            orig_gmt = datetime.strptime(
                f"{row['Date GMT']} {row['Time GMT']}", "%Y-%m-%d %H:%M"
            )
            new_gmt = orig_gmt + time_shift
            row["Date GMT"] = new_gmt.strftime("%Y-%m-%d")
            row["Time GMT"] = new_gmt.strftime("%H:%M")
        except ValueError:
            pass

    return row


def air_quality_transform(row):
    row["aqi"] = pm25_to_aqi(row.get("Sample Measurement", ""))
    return row


source_path = "../data/raw/air_quality/hourly_88101_2024.csv"
update_path = "../data/raw/air_quality/hourly_88101_update.csv"

source_row_count, new_records, duplicates, start_period, end_period = (
    generate_incremental_update(
        source_path=source_path,
        update_path=update_path,
        update_fraction=0.01,
        datetime_builder=air_quality_dt_builder,
        datetime_formatter=air_quality_dt_formatter,
        transform_row=air_quality_transform,
        extra_columns=["aqi"],
        duplicate_fraction=0.01,
    )
)

print("=== DATASET GENERATION REPORT ===")
print(f"Source file:            {source_path}")
print(f"Source records:         {source_row_count:,}")
print(f"Output update file:     {update_path}")
print(f"New records written:    {new_records:,}")
print(f"Duplicates injected:    {duplicates:,}")
print(f"Total update records:   {(new_records + duplicates):,}")
print(f"Schema Evolution:       Added column 'aqi' (Air Quality Index)")
print(f"Time period covered:    {start_period} to {end_period}")

=== DATASET GENERATION REPORT ===
Source file:            ../data/raw/air_quality/hourly_88101_2024.csv
Source records:         8,139,551
Output update file:     ../data/raw/air_quality/hourly_88101_update.csv
New records written:    83,280
Duplicates injected:    833
Total update records:   84,113
Schema Evolution:       Added column 'aqi' (Air Quality Index)
Time period covered:    2025-01-01 00:00:00 to 2025-01-04 15:00:00


### Weather

In [3]:
from datetime import datetime, timedelta

def weather_dt_builder(row: dict) -> datetime:
    try:
        y = int(row["year"])
        m = int(row["month"])
        d = int(row["day"])
        h = int(float(row["hour"]))
        return datetime(y, m, d, h)
    except (KeyError, ValueError, TypeError):
        return None


def weather_dt_formatter(row: dict, new_dt: datetime, time_shift: timedelta) -> dict:
    row["year"] = str(new_dt.year)
    row["month"] = str(new_dt.month)
    row["day"] = str(new_dt.day)
    row["hour"] = str(new_dt.hour)
    return row


def weather_transform(row: dict) -> dict:
    rhum_val = row.get("rhum")
    if rhum_val not in (None, "", "NULL"):
        row["humidity"] = str(rhum_val)
    else:
        row["humidity"] = "60.0"
    return row


source_path = "../data/raw/weather/weather.csv"
update_path = "../data/raw/weather/weather_update.csv"

source_row_count, new_records, duplicates, start_period, end_period = (
    generate_incremental_update(
        source_path=source_path,
        update_path=update_path,
        update_fraction=0.01,
        datetime_builder=weather_dt_builder,
        datetime_formatter=weather_dt_formatter,
        transform_row=weather_transform,
        extra_columns=["humidity"],
        duplicate_fraction=0.01,
    )
)

print("=== DATASET GENERATION REPORT ===")
print(f"Source file:            {source_path}")
print(f"Source records:         {source_row_count:,}")
print(f"Output update file:     {update_path}")
print(f"New records written:    {new_records:,}")
print(f"Duplicates injected:    {duplicates:,}")
print(f"Total update records:   {(new_records + duplicates):,}")
print(f"Schema Evolution:       Added column 'humidity' (Relative Humidity %)")
print(f"Time period covered:    {start_period} to {end_period}")

=== DATASET GENERATION REPORT ===
Source file:            ../data/raw/weather/weather.csv
Source records:         8,784
Output update file:     ../data/raw/weather/weather_update.csv
New records written:    88
Duplicates injected:    1
Total update records:   89
Schema Evolution:       Added column 'humidity' (Relative Humidity %)
Time period covered:    2025-01-01 00:00:00 to 2025-01-04 15:00:00
